#MULTI-AGENT PACKAGE SCORING

###Computer Vision agent

In [ ]:
# ================================================================
# AGENT 1 — COMPUTER VISION AGENT
# JSW Paints · AI Packaging Benchmarking System
# ================================================================
# PURPOSE : Extracts 10 visual features from every package image
#           using OpenCV. Extends the original code with logo
#           prominence, edge sharpness, and dominant colour.
# RUN IN  : Google Colab — paste each cell in a new code cell
# ================================================================

# ── CELL 1 : Install dependencies ───────────────────────────────
!pip install opencv-python-headless pandas numpy openpyxl tqdm -q

# ── CELL 2 : Imports ────────────────────────────────────────────
import os
import cv2
import zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm
from google.colab import files

print("Agent 1 — Computer Vision Agent ready")

# ── CELL 3 : Upload your image ZIP ──────────────────────────────
# HOW TO USE:
#   1. Organise your package images like this BEFORE zipping:
#      Paint_packets/
#        Asian_Paints_Royale.jpg
#        Birla_Opus_Velvet.jpg
#        Dulux_Weathershield.jpg
#        JSW_Aurus_Regal.jpg
#        ... (one image per SKU, named after the brand/product)
#   2. Zip the folder → Paint_packets.zip
#   3. Run this cell → click "Choose Files" → upload the zip

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

EXTRACT_FOLDER = "/content/package_images"
os.makedirs(EXTRACT_FOLDER, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(EXTRACT_FOLDER)

image_paths = []
for root, dirs, files_ in os.walk(EXTRACT_FOLDER):
    for f in files_:
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp')):
            image_paths.append(os.path.join(root, f))

print(f"Images found: {len(image_paths)}")
for p in image_paths:
    print(" →", os.path.basename(p))

# ── CELL 4 : Feature extraction function ────────────────────────
def extract_visual_features(img_path):
    """
    Extracts 10 visual features from a single package image.

    WHAT EACH FEATURE MEANS:
    ┌─────────────────────┬──────────────────────────────────────────┐
    │ Feature             │ What it measures                         │
    ├─────────────────────┼──────────────────────────────────────────┤
    │ Brightness          │ How light/dark the pack is (HSV V channel│
    │ Saturation          │ Colour vividness (HSV S channel)         │
    │ Contrast            │ Visual punch — std dev of grayscale      │
    │ Color Diversity     │ Unique colour count / total pixels       │
    │ Whitespace Ratio    │ % of near-white pixels — premium proxy  │
    │ Edge Sharpness      │ Canny edge density — design crispness   │
    │ Logo Prominence     │ Estimated logo area fraction (top zone) │
    │ Dominant Hue        │ Most common HSV hue — brand colour      │
    │ Structure Score     │ Bucket=8 (premium), Rectangular=6       │
    │ Dark Zone Ratio     │ % dark pixels — premium/luxury signal   │
    └─────────────────────┴──────────────────────────────────────────┘
    """
    img = cv2.imread(img_path)
    if img is None:
        return None

    h, w, _ = img.shape

    # ── Colour space conversions
    img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    hsv      = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # ── Basic visual metrics (from original code)
    brightness      = float(np.mean(hsv[:, :, 2]))
    saturation      = float(np.mean(hsv[:, :, 1]))
    contrast        = float(np.std(gray))
    color_diversity = len(np.unique(img_rgb.reshape(-1, 3), axis=0)) / (h * w)
    whitespace      = float(np.sum(gray > 230)) / (h * w)

    # ── NEW: Edge sharpness (Canny edge density)
    # Measures how crisp the design elements are.
    # High = sharp typography & lines → premium feel
    # Low  = blurry / soft design → lower quality perception
    edges          = cv2.Canny(gray, 50, 150)
    edge_sharpness = float(np.sum(edges > 0)) / (h * w)

    # ── NEW: Logo prominence (top 25% of image)
    # Most paint brands put their logo in the top quarter.
    # We detect the largest contour in that zone as a logo proxy.
    top_zone       = gray[:h // 4, :]
    _, thresh      = cv2.threshold(top_zone, 0, 255,
                                    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _    = cv2.findContours(thresh, cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)
    logo_area      = 0
    if contours:
        largest    = max(contours, key=cv2.contourArea)
        logo_area  = cv2.contourArea(largest)
    logo_prominence = logo_area / ((h // 4) * w + 1e-6)
    logo_prominence = min(logo_prominence, 1.0)

    # ── NEW: Dominant hue (brand colour identity)
    hue_hist       = cv2.calcHist([hsv], [0], None, [36], [0, 180])
    dominant_hue   = float(np.argmax(hue_hist) * 5)  # degrees 0-175

    # ── Structure & dark zone (from original + extended)
    if h > w:
        structure       = "Bucket/Cylindrical"
        structure_score = 8          # cylindrical = premium paint bucket
    else:
        structure       = "Rectangular"
        structure_score = 6

    dark_zone_ratio = float(np.sum(gray < 60)) / (h * w)  # luxury / dark background

    return {
        "Image":            img_path,
        "Brand":            os.path.splitext(os.path.basename(img_path))[0],
        "Width":            w,
        "Height":           h,
        "Brightness":       brightness,
        "Saturation":       saturation,
        "Contrast":         contrast,
        "Color_Diversity":  color_diversity,
        "Whitespace":       whitespace,
        "Edge_Sharpness":   edge_sharpness,
        "Logo_Prominence":  logo_prominence,
        "Dominant_Hue":     dominant_hue,
        "Dark_Zone_Ratio":  dark_zone_ratio,
        "Structure":        structure,
        "Structure_Score":  structure_score,
    }

# ── CELL 5 : Run Agent 1 on all images ──────────────────────────
print("\nAgent 1 running — extracting visual features...")
visual_results = []

for img_path in tqdm(image_paths, desc="Processing images"):
    result = extract_visual_features(img_path)
    if result:
        visual_results.append(result)
    else:
        print(f"  WARNING: Could not read {img_path}")

visual_df = pd.DataFrame(visual_results)

print(f"\nAgent 1 complete — {len(visual_df)} images processed")
print("\nSample output:")
print(visual_df[["Brand", "Brightness", "Saturation", "Contrast",
                  "Whitespace", "Edge_Sharpness", "Logo_Prominence",
                  "Structure"]].to_string(index=False))

# Save Agent 1 output (passed to other agents)
visual_df.to_csv("/content/agent1_visual_features.csv", index=False)
print("\nSaved: /content/agent1_visual_features.csv")
print("Hand this file to Agent 2, 3, and 4.")

Agent 1 — Computer Vision Agent ready


Saving Paint_packets.zip to Paint_packets (1).zip
Images found: 54
 → Screenshot 2026-06-19 161229.png
 → Screenshot 2026-06-19 162006.png
 → Screenshot 2026-06-18 161621.png
 → Screenshot 2026-06-10 140031.png
 → Screenshot 2026-06-18 161457.png
 → Screenshot 2026-06-19 161704.png
 → Screenshot 2026-06-11 125345.png
 → Screenshot 2026-06-19 161133.png
 → Screenshot 2026-06-18 162229.png
 → Screenshot 2026-06-18 161211.png
 → Screenshot 2026-06-18 162001.png
 → Screenshot 2026-06-18 162608.png
 → Screenshot 2026-06-18 162631.png
 → Screenshot 2026-06-18 160144.png
 → Screenshot 2026-06-19 162636.png
 → Screenshot 2026-06-18 161752.png
 → Screenshot 2026-06-19 161324.png
 → Screenshot 2026-06-10 141254.png
 → Screenshot 2026-06-18 161841.png
 → Screenshot 2026-06-18 162707.png
 → Screenshot 2026-06-18 162848.png
 → Screenshot 2026-06-18 162540.png
 → Screenshot 2026-06-10 141222.png
 → Screenshot 2026-06-19 160943.png
 → Screenshot 2026-06-18 162441.png
 → Screenshot 2026-06-18 160028.p

Processing images: 100%|██████████| 54/54 [00:59<00:00,  1.09s/it]


Agent 1 complete — 54 images processed

Sample output:
                       Brand  Brightness  Saturation  Contrast  Whitespace  Edge_Sharpness  Logo_Prominence          Structure
Screenshot 2026-06-19 161229  184.222788  104.711791 67.735661    0.236951        0.061935         0.826249 Bucket/Cylindrical
Screenshot 2026-06-19 162006  141.994283   47.955844 80.992528    0.155370        0.050276         0.278412 Bucket/Cylindrical
Screenshot 2026-06-18 161621  206.948976   85.136703 59.044953    0.167716        0.029366         0.993608 Bucket/Cylindrical
Screenshot 2026-06-10 140031  157.715732   89.553801 63.401005    0.033419        0.037367         0.373862 Bucket/Cylindrical
Screenshot 2026-06-18 161457  158.373134   19.117039 71.632867    0.247291        0.084585         0.118512 Bucket/Cylindrical
Screenshot 2026-06-19 161704  136.808514   53.335045 80.015286    0.153571        0.044306         0.506373 Bucket/Cylindrical
Screenshot 2026-06-11 125345  152.080064   68.144937 81

##OCR Text Agent

In [ ]:
# ================================================================
# AGENT 2 — OCR TEXT AGENT
# JSW Paints · AI Packaging Benchmarking System
# ================================================================
# PURPOSE : Reads all text on every package image using EasyOCR.
#           Measures benefit communication, claims count, hierarchy
#           quality, and text clutter.
# DEPENDS : Run Agent 1 first so /content/agent1_visual_features.csv
#           and image_paths are available.
# ================================================================

# ── CELL 1 : Install EasyOCR ────────────────────────────────────
!pip install easyocr -q

# ── CELL 2 : Imports ────────────────────────────────────────────
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import easyocr
import json

print("Agent 2 — OCR Text Agent ready")
print("Initialising EasyOCR (downloads model on first run ~1 min)...")
reader = easyocr.Reader(['en'], gpu=False)
print("EasyOCR ready")

# ── CELL 3 : Keyword dictionaries ───────────────────────────────
# HOW THIS WORKS:
#   EasyOCR reads every word on the pack.
#   We check how many words match our keyword lists.
#   More relevant keywords = higher communication clarity.

BENEFIT_KEYWORDS = [
    # Performance
    "washable", "anti", "weather", "durable", "coverage",
    "shine", "finish", "resistant", "waterproof", "protection",
    # Quality signals
    "luxury", "premium", "smooth", "stain", "ultra", "super",
    # Technical claims
    "warranty", "certified", "leed", "griha", "isi",
    "lead free", "low voc", "eco", "green",
    # Application
    "interior", "exterior", "emulsion", "primer", "putty",
    # JSW specific
    "one price", "colour change", "fragrant", "any colour"
]

PREMIUM_SIGNALS = [
    "luxury", "royale", "velvet", "silk", "satin",
    "gold", "platinum", "premium", "elegance", "prestige"
]

HIERARCHY_MARKERS = [
    # Words that indicate the pack has clear structure
    "interior", "exterior", "primer", "emulsion", "putty",
    "1l", "4l", "10l", "20l", "1 litre", "4 litre"
]

# ── CELL 4 : OCR extraction function ────────────────────────────
def extract_ocr_features(img_path, reader):
    """
    Reads all text from the package image and scores communication.

    WHAT EACH FEATURE MEANS:
    ┌──────────────────────────┬────────────────────────────────────┐
    │ Feature                  │ What it measures                   │
    ├──────────────────────────┼────────────────────────────────────┤
    │ Text_Count               │ Total number of text blocks found  │
    │ Benefit_Count            │ # benefit/feature keywords found   │
    │ Premium_Signal_Count     │ # luxury/premium words found       │
    │ Hierarchy_Marker_Count   │ # clear navigation words found     │
    │ Text_Density             │ % of pack surface covered by text  │
    │ Avg_Conf                 │ EasyOCR average read confidence    │
    │ Top_Zone_Text_Ratio      │ % of text in top 30% (brand zone) │
    │ Clarity_Score            │ Composite communication quality    │
    │ Clutter_Penalty          │ High text density penalty          │
    │ Full_Text                │ All extracted text (raw)           │
    └──────────────────────────┴────────────────────────────────────┘

    CLARITY SCORE FORMULA:
        benefit_count × 2.0      (each benefit keyword = +2)
      + premium_signal × 1.5     (each premium word = +1.5)
      + hierarchy_marker × 1.5   (each navigation word = +1.5)
      + (1 - text_density) × 4   (reward for breathing room)
      + avg_conf × 2             (reward for readable text)
    """
    try:
        img    = cv2.imread(img_path)
        if img is None:
            return None
        h, w, _ = img.shape
        image_area = h * w

        results = reader.readtext(img_path)

        texts           = []
        confidences     = []
        total_text_area = 0
        top_zone_area   = 0   # text area in top 30% of image

        for item in results:
            bbox, text, conf = item

            texts.append(text.strip())
            confidences.append(conf)

            x1 = int(bbox[0][0]);  y1 = int(bbox[0][1])
            x2 = int(bbox[2][0]);  y2 = int(bbox[2][1])
            area = abs((x2 - x1) * (y2 - y1))
            total_text_area += area

            if y1 < h * 0.30:          # in top 30% of image
                top_zone_area += area

        full_text = " ".join(texts).lower()

        # Keyword scoring
        benefit_count   = sum(kw in full_text for kw in BENEFIT_KEYWORDS)
        premium_count   = sum(kw in full_text for kw in PREMIUM_SIGNALS)
        hierarchy_count = sum(kw in full_text for kw in HIERARCHY_MARKERS)

        text_density        = total_text_area / image_area
        top_zone_ratio      = top_zone_area / (image_area * 0.30 + 1e-6)
        avg_conf            = float(np.mean(confidences)) if confidences else 0.0

        # Clutter: penalise packs where text covers >60% of surface
        clutter_penalty = max(0, text_density - 0.60) * 5

        clarity_score = (
            benefit_count   * 2.0 +
            premium_count   * 1.5 +
            hierarchy_count * 1.5 +
            (1 - text_density) * 4.0 +
            avg_conf * 2.0
        ) - clutter_penalty

        return {
            "Image":                img_path,
            "Brand":                os.path.splitext(os.path.basename(img_path))[0],
            "Text_Count":           len(texts),
            "Benefit_Count":        benefit_count,
            "Premium_Signal_Count": premium_count,
            "Hierarchy_Marker_Count": hierarchy_count,
            "Text_Density":         text_density,
            "Avg_OCR_Conf":         avg_conf,
            "Top_Zone_Text_Ratio":  min(top_zone_ratio, 1.0),
            "Clarity_Score":        max(clarity_score, 0),
            "Clutter_Penalty":      clutter_penalty,
            "Full_Text":            full_text,
        }

    except Exception as e:
        print(f"  OCR error on {img_path}: {e}")
        return None

# ── CELL 5 : Run Agent 2 on all images ──────────────────────────
# NOTE: image_paths must exist from Agent 1's CELL 3.
# If you're running Agent 2 in a fresh session, re-run Agent 1 CELL 3 first.

print("\nAgent 2 running — reading text from packaging...")
print("(EasyOCR is slow — ~20-40 sec per image. This is normal.)\n")

ocr_results = []
for img_path in tqdm(image_paths, desc="OCR scanning"):
    result = extract_ocr_features(img_path, reader)
    if result:
        ocr_results.append(result)

ocr_df = pd.DataFrame(ocr_results)

print(f"\nAgent 2 complete — {len(ocr_df)} images processed")
print("\nSample output (key columns):")
cols = ["Brand","Text_Count","Benefit_Count","Premium_Signal_Count",
        "Text_Density","Clarity_Score","Clutter_Penalty"]
print(ocr_df[cols].to_string(index=False))

# ── CELL 6 : Show extracted text per brand ──────────────────────
print("\n── Extracted text per brand ──────────────────────────────")
for _, row in ocr_df.iterrows():
    print(f"\n{row['Brand']}:")
    print(f"  Words found   : {row['Text_Count']}")
    print(f"  Benefits found: {row['Benefit_Count']} {BENEFIT_KEYWORDS[:3]}...")
    print(f"  Text preview  : {row['Full_Text'][:120]}...")

# Save Agent 2 output
ocr_df.to_csv("/content/agent2_ocr_features.csv", index=False)
print("\nSaved: /content/agent2_ocr_features.csv")
print("Hand this file to Agent 3 and Agent 4.")

Agent 2 — OCR Text Agent ready
Initialising EasyOCR (downloads model on first run ~1 min)...
EasyOCR ready

Agent 2 running — reading text from packaging...
(EasyOCR is slow — ~20-40 sec per image. This is normal.)



OCR scanning:   0%|          | 0/54 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
OCR scanning:   2%|▏         | 1/54 [00:12<11:28, 12.98s/it]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
OCR scanning:   4%|▎         | 2/54 [00:28<12:32, 14.47s/it]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
OCR scanning:   6%|▌         | 3/54 [00:39<11:05, 13.06s/it]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memor


Agent 2 complete — 54 images processed

Sample output (key columns):
                       Brand  Text_Count  Benefit_Count  Premium_Signal_Count  Text_Density  Clarity_Score  Clutter_Penalty
Screenshot 2026-06-19 161229          22              4                     0      0.314957      15.304468                0
Screenshot 2026-06-19 162006          16              0                     0      0.313631       3.917092                0
Screenshot 2026-06-18 161621           3              1                     1      0.191006       7.726653                0
Screenshot 2026-06-10 140031          26              3                     0      0.181538      13.847940                0
Screenshot 2026-06-18 161457           7              0                     0      0.091590       4.745525                0
Screenshot 2026-06-19 161704          17              3                     0      0.194846      13.330970                0
Screenshot 2026-06-11 125345           4              0       

##Packaging Consultant Agent

In [ ]:
# ==============================================================================
# AGENT 3 — LLM PACKAGING CONSULTANT AGENT (PRODUCTION READ REVISED)
# JSW Paints · AI Packaging Benchmarking System
# ==============================================================================

# ── STEP 1 : INSTALL DEPENDENCIES ─────────────────────────────────────────────
# Install the Google GenAI SDK silently
!pip install google-generativeai -q

# ── STEP 2 : SYSTEM IMPORTS ───────────────────────────────────────────────────
import os
import json
import time
import pandas as pd
import numpy as np
import google.generativeai as genai
from PIL import Image

print("[✓] Environment initialized. Agent 3 is ready.")

# ── STEP 3 : API CREDENTIALS & INITIALIZATION ─────────────────────────────────
# Using your active API credentials with native JSON extraction constraints
GEMINI_API_KEY = "AQ.Ab8RN6L8YDBhjaYuZRih_pcILyQnjIJ46Rgnq6ZFePNQH0T_JA"
genai.configure(api_key=GEMINI_API_KEY, client_options={'api_endpoint': 'generativelanguage.googleapis.com'})

# Setting up 1.5 Flash with mandatory JSON output configuration
model = genai.GenerativeModel(
    "gemini-1.5-flash-latest",
    generation_config={"response_mime_type": "application/json"}
)

print("[✓] Gemini model configured: gemini-1.5-flash-latest")
print("[!] Rate limit warning: 15 requests/min enforcement active.")

##Pipelines

In [ ]:
# ── STEP 4 : DATA INGESTION & FEATURE MERGING ─────────────────────────────────
try:
    visual_df = pd.read_csv("/content/agent1_visual_features.csv")
    ocr_df    = pd.read_csv("/content/agent2_ocr_features.csv")

    # Merge execution on unique asset pathways
    combined_df = visual_df.merge(ocr_df, on=["Image", "Brand"], how="left")
    combined_df.fillna(0, inplace=True)

    print(f"[✓] Ingested {len(combined_df)} brand profiles for LLM valuation:")
    for b in combined_df["Brand"].tolist():
        print(f"  → {b}")
except FileNotFoundError as e:
    print(f"[X] Execution halted: Missing pipeline input dependencies. Check upstream agents.\nError: {e}")

# ── STEP 5 : DYNAMIC STRATIFIED PROMPT GENERATOR ──────────────────────────────
def build_prompt(row):
    """
    Normalizes technical CV and OCR telemetry variables into standard ranges
    to drive highly consistent scoring across diverse competitive configurations.
    """
    brightness_norm = min(row["Brightness"] / 255, 1.0)
    saturation_norm = min(row["Saturation"] / 255, 1.0)
    contrast_norm   = min(row["Contrast"]   / 128, 1.0)
    whitespace      = round(float(row["Whitespace"]), 3)
    text_density    = round(float(row["Text_Density"]) if "Text_Density" in row else 0, 3)
    benefit_count   = int(row.get("Benefit_Count", 0))
    premium_signals = int(row.get("Premium_Signal_Count", 0))
    logo_prominence = round(float(row.get("Logo_Prominence", 0)), 3)
    edge_sharpness  = round(float(row.get("Edge_Sharpness", 0)), 3)
    dark_zone       = round(float(row.get("Dark_Zone_Ratio", 0)), 3)
    extracted_text  = str(row.get("Full_Text", ""))[:300]

    prompt = f"""
You are a senior packaging consultant specialising in the Indian decorative paint industry.
Your client is JSW Paints, benchmarking competitor packaging for a relaunch project.

You are evaluating the packaging of: **{row['Brand']}**

MEASURED METRICS (from computer vision and OCR analysis):
- Brightness level      : {brightness_norm:.2f}  (0=very dark, 1=very bright)
- Colour saturation     : {saturation_norm:.2f}  (0=muted, 1=vivid)
- Contrast level        : {contrast_norm:.2f}    (0=flat, 1=high contrast)
- Whitespace ratio      : {whitespace:.2f}       (higher = cleaner, more premium)
- Text density on pack  : {text_density:.2f}     (higher = more cluttered)
- Edge sharpness        : {edge_sharpness:.2f}   (higher = crisper design)
- Logo prominence       : {logo_prominence:.2f}  (higher = logo dominates top zone)
- Dark zone ratio       : {dark_zone:.2f}        (higher = dark/luxury aesthetic)
- Benefit keywords found: {benefit_count}
- Premium signal words  : {premium_signals}
- Pack text (excerpt)   : "{extracted_text}"

SCORING GUIDE:
- Score 9-10 = Category leader. Sets the benchmark. No meaningful gap.
- Score 7-8  = Above average. Clear strength, minor room to improve.
- Score 5-6  = Average. Parity with most competitors.
- Score 3-4  = Below average. Noticeable weakness.
- Score 1-2  = Poor. Significant redesign needed.

TASK:
Evaluate this packaging across 5 dimensions and provide 3 strengths,
3 weaknesses, and 3 actionable recommendations.

Output format must obey the exact structural schema requested below.
{{
  "brand": "{row['Brand']}",
  "premium_score": <number 1-10>,
  "shelf_standout_score": <number 1-10>,
  "communication_score": <number 1-10>,
  "hierarchy_score": <number 1-10>,
  "contractor_appeal_score": <number 1-10>,
  "overall_llm_score": <weighted average of above>,
  "strengths": [
    "<specific strength 1>",
    "<specific strength 2>",
    "<specific strength 3>"
  ],
  "weaknesses": [
    "<specific weakness 1>",
    "<specific weakness 2>",
    "<specific weakness 3>"
  ],
  "recommendations": [
    "<actionable recommendation 1>",
    "<actionable recommendation 2>",
    "<actionable recommendation 3>"
  ],
  "one_line_verdict": "<single sentence summary of this pack's biggest opportunity>"
}}
"""
    return prompt.strip()

[✓] Ingested 54 brand profiles for LLM valuation:
  → Screenshot 2026-06-19 161229
  → Screenshot 2026-06-19 162006
  → Screenshot 2026-06-18 161621
  → Screenshot 2026-06-10 140031
  → Screenshot 2026-06-18 161457
  → Screenshot 2026-06-19 161704
  → Screenshot 2026-06-11 125345
  → Screenshot 2026-06-19 161133
  → Screenshot 2026-06-18 162229
  → Screenshot 2026-06-18 161211
  → Screenshot 2026-06-18 162001
  → Screenshot 2026-06-18 162608
  → Screenshot 2026-06-18 162631
  → Screenshot 2026-06-18 160144
  → Screenshot 2026-06-19 162636
  → Screenshot 2026-06-18 161752
  → Screenshot 2026-06-19 161324
  → Screenshot 2026-06-10 141254
  → Screenshot 2026-06-18 161841
  → Screenshot 2026-06-18 162707
  → Screenshot 2026-06-18 162848
  → Screenshot 2026-06-18 162540
  → Screenshot 2026-06-10 141222
  → Screenshot 2026-06-19 160943
  → Screenshot 2026-06-18 162441
  → Screenshot 2026-06-18 160028
  → Screenshot 2026-06-18 161933
  → Screenshot 2026-06-18 162254
  → Screenshot 2026-06-19 

##Execution

In [ ]:
# ── STEP 6 : API CLIENT WITH FAULT-TOLERANT BACKOFF ───────────────────────────
def call_gemini(prompt, retries=3, delay=5):
    """
    Executes standard inference over the target prompt. Native structured routing
    bypasses the need for markdown parsing or code block validation regex.
    """
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            # Direct deserialization is safe since native JSON mode enforces structure
            return json.loads(response.text.strip())
        except json.JSONDecodeError as e:
            print(f"  [!] Structured syntax variance (attempt {attempt+1}): {e}")
            time.sleep(delay)
        except Exception as e:
            print(f"  [!] System connection error (attempt {attempt+1}): {e}")
            time.sleep(delay * (attempt + 1))

    # Strict fallback data vector mapping
    return {
        "brand": "unknown", "premium_score": 5.0, "shelf_standout_score": 5.0,
        "communication_score": 5.0, "hierarchy_score": 5.0, "contractor_appeal_score": 5.0,
        "overall_llm_score": 5.0, "strengths": ["Processing Exception"],
        "weaknesses": ["Processing Exception"], "recommendations": ["Re-run run instance with stable pipe connection"],
        "one_line_verdict": "Evaluation criteria failed to resolve."
    }

# ── STEP 7 : EXECUTE BATCH PROCESSING PIPELINE ────────────────────────────────
print("\n[Running] Querying Gemini Consultant Engine...")
llm_results = []

for idx, row in combined_df.iterrows():
    brand = row["Brand"]
    print(f"  [{idx+1}/{len(combined_df)}] Processing Matrix: {brand}...", end=" ")

    prompt = build_prompt(row)
    result = call_gemini(prompt)
    result["Image"] = row["Image"] # Maintain strict relational integrity
    llm_results.append(result)

    print(f"✓ Score: {result.get('overall_llm_score', '?')}/10")
    time.sleep(5)  # Safe buffer to respect free tier boundaries (15 RPM)

llm_df = pd.DataFrame(llm_results)

# ── STEP 8 : REPORT GENERATION & EXPORT SYSTEM ────────────────────────────────
print("\n" + "═"*60 + "\nAGENT 3 — EXECUTIVE EVALUATION REPORT\n" + "═"*60)

for _, row in llm_df.iterrows():
    print(f"\n{'-'*60}\nBRAND ASSESSMENT PROFILE: {row['brand'].upper()}\n{'-'*60}")
    print(f"  • Premium Metric Score      : {row['premium_score']}/10")
    print(f"  • Shelf Prominence Score    : {row['shelf_standout_score']}/10")
    print(f"  • Functional Messaging Score: {row['communication_score']}/10")
    print(f"  • Layout Hierarchy Score    : {row['hierarchy_score']}/10")
    print(f"  • Commercial Appeal Score   : {row['contractor_appeal_score']}/10")
    print(f"  • Consolidated Overall Score: {row['overall_llm_score']}/10")
    print(f"\n  Strategic Summary: {row['one_line_verdict']}")
    print(f"\n  Identified Strengths:")
    for s in row['strengths']: print(f"    ✓ {s}")
    print(f"\n  Structural Bottlenecks:")
    for w in row['weaknesses']: print(f"    ✗ {w}")
    print(f"\n  Tactical Interventions:")
    for i, r in enumerate(row['recommendations'], 1): print(f"    {i}. {r}")

# Write standardized output targets
llm_df.to_json("/content/agent3_llm_scores.json", orient="records", indent=2)
llm_df.to_csv("/content/agent3_llm_scores.csv", index=False)
print("\n[✓] Outputs successfully routed to structural JSON and CSV targets. Ready for Agent 4.")

# ── STEP 9 : BONUS MULTIMODAL COMPONENT ───────────────────────────────────────
print("\n\n" + "─"*60 + "\nBONUS MODULE: MULTIMODAL COMPUTER VISION INSPECTION\n" + "─"*60)

# FIXED: Extract directly from data frame tracking array to prevent variable isolation errors
image_paths = combined_df["Image"].tolist()

if image_paths and len(image_paths) > 0:
    first_img_path = image_paths[0]
    try:
        first_img = Image.open(first_img_path)
        brand_name = combined_df["Brand"].iloc[0]
        print(f"[Running] Analyzing spatial properties for initial brand: {brand_name}")

        vision_prompt = f"""
        You are an expert packaging designer specializing in consumer psychology in India.
        Inspect this exact asset representation for: {brand_name}

        Evaluate these 5 variables dynamically on a 1-10 range:
        1. Luxury Appeal
        2. Shelf Visibility
        3. Communication
        4. Modernity
        5. Contractor Appeal

        Output strict JSON structure using these exact keys:
        {{
          "luxury_appeal": <1-10>,
          "shelf_visibility": <1-10>,
          "communication": <1-10>,
          "modernity": <1-10>,
          "contractor_appeal": <1-10>,
          "top_visual_strength": "<one sentence>",
          "top_visual_weakness": "<one sentence>"
        }}
        """
        vision_result = model.generate_content([vision_prompt, first_img])
        print(f"\n[✓] Direct Canvas Interpretation Results ({brand_name}):")
        print(vision_result.text)
    except Exception as e:
        print(f"[X] Multimodal evaluation skipped. Pipeline file exception at: {first_img_path}\nError: {e}")
else:
    print("[!] Processing array empty. Confirm target paths exist within processing frame.")


[Running] Querying Gemini Consultant Engine...
  [1/54] Processing Matrix: Screenshot 2026-06-19 161229... 

  [!] System connection error (attempt 1): 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.


  [!] System connection error (attempt 2): 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.


  [!] System connection error (attempt 3): 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
✓ Score: 5.0/10
  [2/54] Processing Matrix: Screenshot 2026-06-19 162006... 

  [!] System connection error (attempt 1): 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.


...

  [!] System connection error (attempt 1): 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.


  [!] System connection error (attempt 2): 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.


  [!] System connection error (attempt 3): 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
✓ Score: 5.0/10
  [40/54] Processing Matrix: Screenshot 2026-06-18 162318... 

  [!] System connection error (attempt 1): 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.


  [!] System connection error (attempt 2): 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
  [!] System connection error (attempt 3): HTTPConnectionPool(host='localhost', port=36425): Read timed out. (read timeout=600.0)
✓ Score: 5.0/10
  [41/54] Processing Matrix: Screenshot 2026-06-19 163130...   [!] System connection error (attempt 1): HTTPConnectionPool(host='localhost', port=36425): Read timed out. (read timeout=600.0)
  [!] System connection error (attempt 2): HTTPConnectionPool(host='localhost', port=36425): Read timed out. (read timeout=600.0)
  [!] System connection error (attempt 3): HTTPConnectionPool(host='localhost', port=36425): Read timed out. (read timeout=600.0)
✓ Score: 5.0/10
  [42/54] Proce

##Benchmarking Agent

In [ ]:
# ================================================================
# AGENT 4 — BENCHMARKING AGENT
# JSW Paints · AI Packaging Benchmarking System
# ================================================================
# PURPOSE : Collects all outputs from Agents 1, 2, 3.
#           Normalises scores, computes composite PEI,
#           ranks all brands, generates a comparison narrative,
#           runs gap analysis for JSW, and exports to Excel.
# DEPENDS : Agents 1, 2, 3 must have run and saved their CSVs/JSONs.
# ================================================================

# ── CELL 1 : Imports ────────────────────────────────────────────
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print("Agent 4 — Benchmarking Agent ready")

# ── CELL 2 : Load all agent outputs ─────────────────────────────
print("\nLoading agent outputs...")

visual_df = pd.read_csv("/content/agent1_visual_features.csv")
ocr_df    = pd.read_csv("/content/agent2_ocr_features.csv")
llm_df    = pd.read_csv("/content/agent3_llm_scores.csv")

# Merge everything on Brand name
df = visual_df.merge(ocr_df,  on=["Image", "Brand"], how="left")
df = df.merge(
    llm_df[["Image","premium_score","shelf_standout_score",
            "communication_score","hierarchy_score",
            "contractor_appeal_score","overall_llm_score",
            "strengths","weaknesses","recommendations","one_line_verdict"]],
    on="Image",
    how="left"
)
df.fillna(0, inplace=True)

print(f"Merged dataset: {len(df)} brands")
print("Brands in dataset:", df["Brand"].tolist())

# ── CELL 3 : Normalise raw features ─────────────────────────────
# MinMaxScaler converts all raw metrics to 0-1 range
# so they can be combined fairly regardless of their original scale.
# Brightness (0-255) and Color_Diversity (0-0.001) would otherwise
# have wildly different influences on the composite score.

raw_metrics = [
    "Brightness", "Saturation", "Contrast", "Color_Diversity",
    "Whitespace", "Edge_Sharpness", "Logo_Prominence",
    "Dark_Zone_Ratio", "Structure_Score",
    "Benefit_Count", "Premium_Signal_Count",
    "Text_Density", "Clarity_Score",
]

# Only normalise columns that actually exist in the merged df
existing_metrics = [m for m in raw_metrics if m in df.columns]

scaler = MinMaxScaler()
df_norm = df.copy()
df_norm[existing_metrics] = scaler.fit_transform(df[existing_metrics])

print(f"\nNormalised {len(existing_metrics)} raw metrics to 0-1 scale")

# ── CELL 4 : Component score calculation ────────────────────────
# ────────────────────────────────────────────────────────────────
# SCORE ARCHITECTURE:
#
#  VISIBILITY SCORE (0-10)
#  = measures how much the pack stands out on a shelf
#    → Brightness + Saturation + Contrast + Edge_Sharpness
#
#  PREMIUM SCORE (0-10)
#  = measures how expensive/aspirational the pack looks
#    → Whitespace + Contrast + Color_Diversity + Structure
#    → Premium_Signal_Count + Dark_Zone_Ratio
#
#  CLARITY SCORE (0-10)
#  = measures how well the pack communicates its benefits
#    → Benefit_Count + Clarity_Score + (1 - Text_Density)
#
#  HIERARCHY SCORE (0-10)
#  = measures how easy it is to navigate the information
#    → (1 - Text_Density) + Logo_Prominence
#
#  MODERNITY SCORE (0-10)
#  = measures how contemporary the design feels
#    → Whitespace + Color_Diversity + Edge_Sharpness
#
#  LLM SCORE (0-10)
#  = Gemini's overall qualitative assessment
#    → overall_llm_score (already 0-10 from Agent 3)
#
#  FINAL PEI  (0-10)
#  = weighted composite of all 6 component scores
# ────────────────────────────────────────────────────────────────

def safe(col, df=df_norm):
    return df[col] if col in df.columns else pd.Series(np.zeros(len(df)))

# Component scores
df_norm["Score_Visibility"] = (
    0.30 * safe("Brightness")    +
    0.25 * safe("Saturation")    +
    0.30 * safe("Contrast")      +
    0.15 * safe("Edge_Sharpness")
) * 10

df_norm["Score_Premium"] = (
    0.25 * safe("Whitespace")           +
    0.20 * safe("Contrast")             +
    0.20 * safe("Color_Diversity")      +
    0.15 * safe("Structure_Score")      +
    0.10 * safe("Premium_Signal_Count") +
    0.10 * safe("Dark_Zone_Ratio")
) * 10

df_norm["Score_Clarity"] = (
    0.40 * safe("Benefit_Count")   +
    0.40 * safe("Clarity_Score")   +
    0.20 * (1 - safe("Text_Density"))
) * 10

df_norm["Score_Hierarchy"] = (
    0.60 * (1 - safe("Text_Density")) +
    0.40 * safe("Logo_Prominence")
) * 10

df_norm["Score_Modernity"] = (
    0.40 * safe("Whitespace")       +
    0.30 * safe("Color_Diversity")  +
    0.30 * safe("Edge_Sharpness")
) * 10

df_norm["Score_LLM"] = df["overall_llm_score"].astype(float)

# ── CELL 5 : Composite PEI ──────────────────────────────────────
# PEI WEIGHT RATIONALE:
# Premium (0.25) — highest weight because at-shelf purchase decisions
#   in the paint category are strongly driven by perceived quality.
# Clarity (0.20) — dealer ease-of-recommendation is the #1 commercial lever.
# Visibility (0.20) — shelf standout drives initial trial.
# LLM Score (0.15) — qualitative consultant view balances numeric metrics.
# Hierarchy (0.10) — navigation reduces dealer explanation time.
# Modernity (0.10) — contemporary design signals innovation.

df_norm["PEI"] = (
    0.20 * df_norm["Score_Visibility"] +
    0.25 * df_norm["Score_Premium"]    +
    0.20 * df_norm["Score_Clarity"]    +
    0.10 * df_norm["Score_Hierarchy"]  +
    0.10 * df_norm["Score_Modernity"]  +
    0.15 * df_norm["Score_LLM"]
)

# Restore brand info
df_norm["Brand"]          = df["Brand"]
df_norm["Image"]          = df["Image"]
df_norm["strengths"]      = df["strengths"]
df_norm["weaknesses"]     = df["weaknesses"]
df_norm["recommendations"]= df["recommendations"]
df_norm["one_line_verdict"]= df["one_line_verdict"]

# Sort by PEI descending
df_norm = df_norm.sort_values("PEI", ascending=False).reset_index(drop=True)
df_norm["Rank"] = range(1, len(df_norm) + 1)

print("\nPEI Rankings:")
score_cols = ["Rank","Brand","Score_Visibility","Score_Premium",
              "Score_Clarity","Score_Hierarchy","Score_Modernity",
              "Score_LLM","PEI"]
print(df_norm[score_cols].to_string(index=False))

# ── CELL 6 : Gap analysis for JSW Paints ─────────────────────────
print("\n\n══════════════════════════════════════════════════")
print("JSW PAINTS — GAP ANALYSIS vs. MARKET LEADER")
print("══════════════════════════════════════════════════")

score_dims = {
    "Visibility" : "Score_Visibility",
    "Premium"    : "Score_Premium",
    "Clarity"    : "Score_Clarity",
    "Hierarchy"  : "Score_Hierarchy",
    "Modernity"  : "Score_Modernity",
    "LLM Score"  : "Score_LLM",
    "PEI"        : "PEI",
}

# Find JSW row
jsw_rows  = df_norm[df_norm["Brand"].str.contains("JSW", case=False, na=False)]
top_row   = df_norm.iloc[0]

if len(jsw_rows) > 0:
    jsw_row = jsw_rows.iloc[0]
    print(f"\nMarket Leader : {top_row['Brand']} (Rank #1)")
    print(f"JSW Paints    : Rank #{jsw_row['Rank']}\n")
    print(f"{'Dimension':<15}  {'Leader':>8}  {'JSW':>8}  {'Gap':>8}  {'Priority'}")
    print("─" * 58)

    gaps = []
    for dim_name, col in score_dims.items():
        leader_val = float(top_row[col])
        jsw_val    = float(jsw_row[col])
        gap        = leader_val - jsw_val
        priority   = "🔴 URGENT" if gap > 2 else ("🟡 MEDIUM" if gap > 1 else "🟢 OK")
        print(f"{dim_name:<15}  {leader_val:>8.2f}  {jsw_val:>8.2f}  {gap:>+8.2f}  {priority}")
        gaps.append({"Dimension": dim_name, "Leader": leader_val,
                     "JSW": jsw_val, "Gap": gap})

    gap_df = pd.DataFrame(gaps).sort_values("Gap", ascending=False)
    print(f"\nBiggest gap for JSW: {gap_df.iloc[0]['Dimension']} "
          f"({gap_df.iloc[0]['Gap']:+.2f})")
    print(f"Smallest gap      : {gap_df.iloc[-1]['Dimension']} "
          f"({gap_df.iloc[-1]['Gap']:+.2f})")
else:
    print("No JSW Paints brand found in the image set.")
    print("Name your JSW image file: JSW_Aurus_Regal.jpg or similar.")

# ── CELL 7 : Visualisations ──────────────────────────────────────
print("\n\nGenerating charts...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("JSW Paints — AI Packaging Benchmarking Dashboard",
             fontsize=15, fontweight="bold", y=0.98)

BRAND_COLORS = {
    "Asian Paints": "#E63946",
    "Birla Opus":   "#F4A261",
    "Dulux":        "#457B9D",
    "Nerolac":      "#6A994E",
    "JSW":          "#1D3557",
}

def brand_color(name):
    for k, v in BRAND_COLORS.items():
        if k.lower() in name.lower():
            return v
    return "#888888"

colors = [brand_color(b) for b in df_norm["Brand"]]

# Chart 1 — PEI ranking bar chart
ax1 = axes[0, 0]
bars = ax1.barh(df_norm["Brand"], df_norm["PEI"], color=colors, alpha=0.85, height=0.6)
ax1.set_xlabel("Packaging Effectiveness Index (PEI)", fontsize=10)
ax1.set_title("Overall PEI Ranking", fontsize=11, fontweight="bold")
ax1.set_xlim(0, 10)
ax1.axvline(x=8.0, color="#DC2626", linestyle="--", lw=1, alpha=0.7, label="Target: 8.0")
ax1.axvline(x=6.0, color="#F59E0B", linestyle="--", lw=1, alpha=0.7, label="Avg: 6.0")
ax1.legend(fontsize=8)
for bar, val in zip(bars, df_norm["PEI"]):
    ax1.text(val + 0.1, bar.get_y() + bar.get_height()/2,
             f"{val:.2f}", va="center", fontsize=9, fontweight="bold")
ax1.spines[["top","right"]].set_visible(False)

# Chart 2 — Component radar (bar chart for simplicity — radar needs >3 brands)
ax2 = axes[0, 1]
component_cols = ["Score_Visibility","Score_Premium","Score_Clarity",
                  "Score_Hierarchy","Score_Modernity","Score_LLM"]
component_labels = ["Visibility","Premium","Clarity","Hierarchy","Modernity","LLM"]
x = np.arange(len(component_labels))
width = 0.8 / max(len(df_norm), 1)

for i, (_, row) in enumerate(df_norm.iterrows()):
    vals   = [float(row[c]) for c in component_cols]
    offset = (i - len(df_norm)/2 + 0.5) * width
    ax2.bar(x + offset, vals, width=width*0.9,
            label=row["Brand"], color=brand_color(row["Brand"]), alpha=0.8)

ax2.set_xticks(x)
ax2.set_xticklabels(component_labels, fontsize=9)
ax2.set_ylabel("Score (0-10)")
ax2.set_title("Component Scores by Brand", fontsize=11, fontweight="bold")
ax2.legend(fontsize=8, bbox_to_anchor=(1,1))
ax2.spines[["top","right"]].set_visible(False)

# Chart 3 — Correlation heatmap
ax3 = axes[1, 0]
heat_cols = ["Score_Visibility","Score_Premium","Score_Clarity",
             "Score_Hierarchy","Score_Modernity","Score_LLM","PEI"]
corr = df_norm[heat_cols].corr()
corr.columns = component_labels + ["PEI"]
corr.index   = component_labels + ["PEI"]
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues",
            ax=ax3, cbar=True, square=True,
            annot_kws={"size": 8}, linewidths=0.5)
ax3.set_title("Score Correlation Matrix", fontsize=11, fontweight="bold")
ax3.tick_params(axis="x", rotation=30, labelsize=8)
ax3.tick_params(axis="y", rotation=0,  labelsize=8)

# Chart 4 — Gap analysis (if JSW exists)
ax4 = axes[1, 1]
if len(jsw_rows) > 0:
    gap_df_plot = gap_df[gap_df["Dimension"] != "PEI"].copy()
    bar_colors  = ["#DC2626" if g > 2 else "#F59E0B" if g > 1 else "#16A34A"
                   for g in gap_df_plot["Gap"]]
    ax4.barh(gap_df_plot["Dimension"], gap_df_plot["Gap"],
             color=bar_colors, alpha=0.85, height=0.55)
    ax4.axvline(x=0, color="black", lw=0.8)
    ax4.set_xlabel("Gap (Leader score − JSW score)")
    ax4.set_title(f"JSW Gap vs. Market Leader ({top_row['Brand']})",
                  fontsize=11, fontweight="bold")
    ax4.spines[["top","right"]].set_visible(False)

    red_p   = mpatches.Patch(color="#DC2626", label="> 2 pts — Urgent")
    amber_p = mpatches.Patch(color="#F59E0B", label="1-2 pts — Medium")
    green_p = mpatches.Patch(color="#16A34A", label="< 1 pt  — OK")
    ax4.legend(handles=[red_p, amber_p, green_p], fontsize=8)
else:
    ax4.text(0.5, 0.5, "Add JSW image to\nsee gap analysis",
             ha="center", va="center", transform=ax4.transAxes,
             fontsize=12, color="gray")

plt.tight_layout()
plt.savefig("/content/PEI_Dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /content/PEI_Dashboard.png")

# ── CELL 8 : Export to formatted Excel ──────────────────────────
OUTPUT_EXCEL = "/content/AI_Packaging_Benchmark_Final.xlsx"

export_cols = [
    "Rank","Brand",
    "Score_Visibility","Score_Premium","Score_Clarity",
    "Score_Hierarchy","Score_Modernity","Score_LLM","PEI",
    "strengths","weaknesses","recommendations","one_line_verdict"
]
export_df = df_norm[[c for c in export_cols if c in df_norm.columns]].copy()
export_df.columns = [c.replace("Score_","") for c in export_df.columns]

# Round numeric columns
num_cols = ["Visibility","Premium","Clarity","Hierarchy","Modernity","LLM","PEI"]
for c in num_cols:
    if c in export_df.columns:
        export_df[c] = export_df[c].round(2)

export_df.to_excel(OUTPUT_EXCEL, index=False)

# Apply minimal formatting with openpyxl
wb   = load_workbook(OUTPUT_EXCEL)
ws   = wb.active
NAVY = "1B3A6B"
LGRY = "F1F5F9"
WHIT = "FFFFFF"

def bdr():
    s = Side(style="thin", color="D1D5DB")
    return Border(left=s, right=s, top=s, bottom=s)

# Header row
for col in range(1, ws.max_column + 1):
    c = ws.cell(row=1, column=col)
    c.font      = Font(name="Arial", bold=True, size=9, color=WHIT)
    c.fill      = PatternFill("solid", fgColor=NAVY)
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    c.border    = bdr()
ws.row_dimensions[1].height = 28

# Data rows
for row_idx in range(2, ws.max_row + 1):
    alt = row_idx % 2 == 0
    for col_idx in range(1, ws.max_column + 1):
        c = ws.cell(row=row_idx, column=col_idx)
        c.font      = Font(name="Arial", size=9)
        c.fill      = PatternFill("solid", fgColor=LGRY if alt else WHIT)
        c.alignment = Alignment(horizontal="left", vertical="center",
                                wrap_text=True)
        c.border    = bdr()
    ws.row_dimensions[row_idx].height = 40

# Column widths
col_widths = [6, 25, 10, 10, 10, 10, 10, 10, 10, 35, 35, 35, 40]
for i, w in enumerate(col_widths[:ws.max_column], 1):
    ws.column_dimensions[get_column_letter(i)].width = w

wb.save(OUTPUT_EXCEL)
print(f"\nSaved: {OUTPUT_EXCEL}")

# Download files
from google.colab import files
files.download(OUTPUT_EXCEL)
files.download("/content/PEI_Dashboard.png")

print("\nAgent 4 complete. Files downloaded to your computer.")
print("\nFinal PEI Rankings:")
print(df_norm[["Rank","Brand","PEI","one_line_verdict"]].to_string(index=False))

#EXTRAAAA

In [ ]:
# ================================================================
# AGENT 3 — LLM PACKAGING CONSULTANT AGENT
# JSW Paints · AI Packaging Benchmarking System
# ================================================================
# PURPOSE : Sends the numeric feature vector from Agents 1+2 to
#           Google Gemini API. Gemini acts as a packaging consultant
#           and returns structured JSON: scores + narrative.
# REQUIRES: Free Google Gemini API key
#           Get it at: https://aistudio.google.com/app/apikey
# ================================================================

# ── CELL 1 : Install Gemini SDK ─────────────────────────────────
!pip install google-generativeai -q

# ── CELL 2 : Imports ────────────────────────────────────────────
import os
import json
import time
import textwrap
import pandas as pd
import numpy as np
import google.generativeai as genai
from PIL import Image

print("Agent 3 — LLM Packaging Consultant ready")

# ── CELL 3 : Configure Gemini API ───────────────────────────────
# HOW TO GET YOUR FREE API KEY:
#   1. Go to https://aistudio.google.com/app/apikey
#   2. Sign in with your Google account
#   3. Click "Create API key"
#   4. Copy the key and paste it below (replacing the placeholder)
#   NOTE: Free tier = 15 requests/min, 1500/day — more than enough

GEMINI_API_KEY = "OPENAI_API_KEY"  # ← paste your key here

genai.configure(api_key=GEMINI_API_KEY)

# Use Gemini 1.5 Flash — free, fast, understands images
model = genai.GenerativeModel("gemini-1.5-flash")

print("Gemini model configured: gemini-1.5-flash")
print("Rate limit: 15 requests/min (free tier)")

# ── CELL 4 : Load Agent 1 + 2 outputs ───────────────────────────
visual_df = pd.read_csv("/content/agent1_visual_features.csv")
ocr_df    = pd.read_csv("/content/agent2_ocr_features.csv")

# Merge on Image path
combined_df = visual_df.merge(ocr_df, on=["Image", "Brand"], how="left")
combined_df.fillna(0, inplace=True)

print(f"Loaded {len(combined_df)} brands for LLM evaluation:")
for b in combined_df["Brand"].tolist():
    print(f"  → {b}")

# ── CELL 5 : Prompt engineering ─────────────────────────────────
def build_prompt(row):
    """
    Builds the prompt sent to Gemini for one package.

    PROMPT DESIGN PRINCIPLES:
    1. Role assignment  — "You are a senior packaging consultant"
       This anchors Gemini's response style and vocabulary.
    2. Numeric grounding — we give Gemini the actual measured values
       so it can't hallucinate scores that contradict the data.
    3. Domain context   — Indian decorative paint market framing
       means Gemini evaluates against the right competitive set.
    4. Strict JSON output — easier to parse programmatically.
    5. Score anchoring  — we define what 10, 7, and 4 mean
       to ensure consistent calibration across brands.
    """

    # Normalise raw values to 0-1 scale for cleaner prompting
    brightness_norm = min(row["Brightness"] / 255, 1.0)
    saturation_norm = min(row["Saturation"] / 255, 1.0)
    contrast_norm   = min(row["Contrast"]   / 128, 1.0)
    whitespace      = round(float(row["Whitespace"]), 3)
    text_density    = round(float(row["Text_Density"]) if "Text_Density" in row else 0, 3)
    benefit_count   = int(row.get("Benefit_Count", 0))
    premium_signals = int(row.get("Premium_Signal_Count", 0))
    logo_prominence = round(float(row.get("Logo_Prominence", 0)), 3)
    edge_sharpness  = round(float(row.get("Edge_Sharpness", 0)), 3)
    dark_zone       = round(float(row.get("Dark_Zone_Ratio", 0)), 3)
    extracted_text  = str(row.get("Full_Text", ""))[:300]  # first 300 chars

    prompt = f"""
You are a senior packaging consultant specialising in the Indian decorative paint industry.
Your client is JSW Paints, benchmarking competitor packaging for a relaunch project.

You are evaluating the packaging of: **{row['Brand']}**

MEASURED METRICS (from computer vision and OCR analysis):
- Brightness level      : {brightness_norm:.2f}  (0=very dark, 1=very bright)
- Colour saturation     : {saturation_norm:.2f}  (0=muted, 1=vivid)
- Contrast level        : {contrast_norm:.2f}    (0=flat, 1=high contrast)
- Whitespace ratio      : {whitespace:.2f}       (higher = cleaner, more premium)
- Text density on pack  : {text_density:.2f}     (higher = more cluttered)
- Edge sharpness        : {edge_sharpness:.2f}   (higher = crisper design)
- Logo prominence       : {logo_prominence:.2f}  (higher = logo dominates top zone)
- Dark zone ratio       : {dark_zone:.2f}        (higher = dark/luxury aesthetic)
- Benefit keywords found: {benefit_count}
- Premium signal words  : {premium_signals}
- Pack text (excerpt)   : "{extracted_text}"

SCORING GUIDE:
- Score 9-10 = Category leader. Sets the benchmark. No meaningful gap.
- Score 7-8  = Above average. Clear strength, minor room to improve.
- Score 5-6  = Average. Parity with most competitors.
- Score 3-4  = Below average. Noticeable weakness.
- Score 1-2  = Poor. Significant redesign needed.

TASK:
Evaluate this packaging across 5 dimensions and provide 3 strengths,
3 weaknesses, and 3 actionable recommendations.

IMPORTANT: Return ONLY valid JSON. No markdown, no backticks, no explanation.
Use this exact structure:

{{
  "brand": "{row['Brand']}",
  "premium_score": <number 1-10>,
  "shelf_standout_score": <number 1-10>,
  "communication_score": <number 1-10>,
  "hierarchy_score": <number 1-10>,
  "contractor_appeal_score": <number 1-10>,
  "overall_llm_score": <weighted average of above>,
  "strengths": [
    "<specific strength 1>",
    "<specific strength 2>",
    "<specific strength 3>"
  ],
  "weaknesses": [
    "<specific weakness 1>",
    "<specific weakness 2>",
    "<specific weakness 3>"
  ],
  "recommendations": [
    "<actionable recommendation 1>",
    "<actionable recommendation 2>",
    "<actionable recommendation 3>"
  ],
  "one_line_verdict": "<single sentence summary of this pack's biggest opportunity>"
}}
"""
    return prompt.strip()

# ── CELL 6 : Call Gemini for each brand ─────────────────────────
def call_gemini(prompt, retries=3, delay=5):
    """
    Calls Gemini API with retry logic for rate limit handling.
    Free tier: 15 requests per minute.
    We add a 5-second delay between calls to stay safe.
    """
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            raw      = response.text.strip()

            # Strip markdown code blocks if Gemini wraps in ```json ... ```
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
            raw = raw.strip()

            parsed = json.loads(raw)
            return parsed

        except json.JSONDecodeError as e:
            print(f"  JSON parse error (attempt {attempt+1}): {e}")
            print(f"  Raw response: {raw[:200]}")
        except Exception as e:
            print(f"  API error (attempt {attempt+1}): {e}")
            time.sleep(delay * (attempt + 1))

    # Fallback if all attempts fail
    return {
        "brand": "unknown",
        "premium_score": 5.0,
        "shelf_standout_score": 5.0,
        "communication_score": 5.0,
        "hierarchy_score": 5.0,
        "contractor_appeal_score": 5.0,
        "overall_llm_score": 5.0,
        "strengths": ["Could not evaluate — API error"],
        "weaknesses": ["Could not evaluate — API error"],
        "recommendations": ["Retry with valid API key"],
        "one_line_verdict": "Evaluation failed."
    }

# ── CELL 7 : Run Agent 3 ────────────────────────────────────────
print("\nAgent 3 running — sending each brand to Gemini...")
print("Rate limiting: 5-second pause between API calls\n")

llm_results = []

for idx, row in combined_df.iterrows():
    brand = row["Brand"]
    print(f"  [{idx+1}/{len(combined_df)}] Evaluating: {brand}...", end=" ")

    prompt   = build_prompt(row)
    result   = call_gemini(prompt)
    result["Image"] = row["Image"]   # keep the link back to the image path
    llm_results.append(result)

    print(f"✓  LLM Score: {result.get('overall_llm_score', '?')}/10")
    time.sleep(5)   # respect free-tier rate limit (15 req/min)

llm_df = pd.DataFrame(llm_results)

# ── CELL 8 : Display results ─────────────────────────────────────
print("\n\n══════════════════════════════════════════════════")
print("AGENT 3 — LLM PACKAGING ASSESSMENTS")
print("══════════════════════════════════════════════════")

for _, row in llm_df.iterrows():
    print(f"\n{'─'*50}")
    print(f"BRAND : {row['brand']}")
    print(f"{'─'*50}")
    print(f"  Premium Score        : {row['premium_score']}/10")
    print(f"  Shelf Standout       : {row['shelf_standout_score']}/10")
    print(f"  Communication Score  : {row['communication_score']}/10")
    print(f"  Hierarchy Score      : {row['hierarchy_score']}/10")
    print(f"  Contractor Appeal    : {row['contractor_appeal_score']}/10")
    print(f"  Overall LLM Score    : {row['overall_llm_score']}/10")
    print(f"\n  Verdict: {row['one_line_verdict']}")
    print(f"\n  STRENGTHS:")
    for s in row['strengths']:
        print(f"    ✓ {s}")
    print(f"\n  WEAKNESSES:")
    for w in row['weaknesses']:
        print(f"    ✗ {w}")
    print(f"\n  RECOMMENDATIONS:")
    for i, r in enumerate(row['recommendations'], 1):
        print(f"    {i}. {r}")

# Save Agent 3 output
llm_df.to_json("/content/agent3_llm_scores.json", orient="records", indent=2)
llm_df.to_csv("/content/agent3_llm_scores.csv",   index=False)
print("\nSaved: /content/agent3_llm_scores.json")
print("Saved: /content/agent3_llm_scores.csv")
print("Hand both files to Agent 4.")

# ── CELL 9 : BONUS — Gemini Vision (analyse the actual image) ───
# This uses Gemini's multimodal capability — sends the IMAGE directly
# without needing OpenCV. Very useful for visual quality evaluation.

print("\n\n── BONUS: Gemini Vision Mode (direct image analysis) ──────")
print("Sending first image directly to Gemini Vision...")

if image_paths:
    first_img_path = image_paths[0]
    first_img      = Image.open(first_img_path)
    brand_name     = os.path.splitext(os.path.basename(first_img_path))[0]

    vision_prompt = f"""
You are a packaging consultant for the Indian decorative paint market.
Look at this paint packaging image for: {brand_name}

Evaluate it on these 5 dimensions (score 1-10 each):
1. Luxury Appeal     — Does it look premium and aspirational?
2. Shelf Visibility  — Would it stand out on a crowded paint shop shelf?
3. Communication     — Are the key benefits immediately clear?
4. Modernity         — Does the design feel contemporary?
5. Contractor Appeal — Would a professional painter trust this product?

Return ONLY JSON with this structure:
{{
  "luxury_appeal": <1-10>,
  "shelf_visibility": <1-10>,
  "communication": <1-10>,
  "modernity": <1-10>,
  "contractor_appeal": <1-10>,
  "top_visual_strength": "<one sentence>",
  "top_visual_weakness": "<one sentence>"
}}
"""
    vision_result = model.generate_content([vision_prompt, first_img])
    print(f"\nGemini Vision output for {brand_name}:")
    print(vision_result.text)

Agent 3 — LLM Packaging Consultant ready
Gemini model configured: gemini-1.5-flash
Rate limit: 15 requests/min (free tier)
Loaded 54 brands for LLM evaluation:
  → Screenshot 2026-06-19 161229
  → Screenshot 2026-06-19 162006
  → Screenshot 2026-06-18 161621
  → Screenshot 2026-06-10 140031
  → Screenshot 2026-06-18 161457
  → Screenshot 2026-06-19 161704
  → Screenshot 2026-06-11 125345
  → Screenshot 2026-06-19 161133
  → Screenshot 2026-06-18 162229
  → Screenshot 2026-06-18 161211
  → Screenshot 2026-06-18 162001
  → Screenshot 2026-06-18 162608
  → Screenshot 2026-06-18 162631
  → Screenshot 2026-06-18 160144
  → Screenshot 2026-06-19 162636
  → Screenshot 2026-06-18 161752
  → Screenshot 2026-06-19 161324
  → Screenshot 2026-06-10 141254
  → Screenshot 2026-06-18 161841
  → Screenshot 2026-06-18 162707
  → Screenshot 2026-06-18 162848
  → Screenshot 2026-06-18 162540
  → Screenshot 2026-06-10 141222
  → Screenshot 2026-06-19 160943
  → Screenshot 2026-06-18 162441
  → Screenshot 

KeyboardInterrupt: 